# Ontario Gateway - Insurance Proposal Analysis


# Executive Summary

**Problem:** Mary Litton, COO of Ontario Gateway, must choose one of four insurance plans (RCNC1, RCNC2, CTC, HIC) for the 86-aircraft fleet for the next five years. Her rules: the first-year cost (crash losses plus insurance cost) must not exceed \$41M, the chance of exceeding \$37M should be as small as possible, and among the plans that meet this, the five-year cost should be the lowest.

**Method:** We built a Monte Carlo simulation in Python with 100,000 five-year futures. In each year, crashes are random (about 1 in 5,000,000 flights, with the aircraft type chosen by its share of flights), and incidental damage of \$1M to \$5M is added. We then calculated what each plan would cost in the same futures, so the plans can be compared fairly.

**Recommendation: choose HIC (Highlands Insurance Company).**

- HIC is the only plan that fully meets the first-year limits. Its first-year cost cannot exceed about \$33.84M (premium plus the \$24M that Ontario Gateway keeps), below both \$37M and \$41M. RCNC2 fails, with a 2.486% chance of exceeding \$41M in year one.
- HIC has the lowest five-year cost among the safe plans: \$65.20M on average, versus \$67.54M for CTC and \$113.68M for RCNC1. HIC beats CTC in 85.7% of simulated futures (average saving \$2.34M) and RCNC1 in 100% (average saving \$48.48M).
- The result holds if Ontario Gateway is 25% safer than the industry: HIC still wins, with an average five-year cost of \$64.51M and savings of \$2.76M against CTC and \$48.62M against RCNC1.

**Main risks:** HIC's advantage over CTC is small, so the main reason to choose it is the guaranteed first-year cap rather than the average saving. Ontario Gateway also pays the first \$24M of losses each year. Our results assume the \$24M applies to total annual losses, that incidental damage is insured, and that costs are not discounted; the contract wording should be checked before signing.


## Assignments


(a) Run the simulation model OGSIM.XLS built for this insurance decision. Make
sure you understand how the model works, especially because you will want to
modify the model to do your analysis.


## Set up


In [17]:
# Author: Zixiao Ma
import pandas as pd
import numpy as np

np.random.seed(123)
# Data input
operating_days = 342
# Boeing 757
boe_num = 47
boe_cost = 56.4
daily_boe_flights = 6
boe_flights = boe_num * daily_boe_flights * operating_days
boe_value = boe_num * boe_cost
# Airbus A340 Model 200
a340_200_num = 15
a340_200_cost = 78.9
daily_a340_200_flights = 2.25
a340_200_flights = a340_200_num * daily_a340_200_flights * operating_days
a340_200_value = a340_200_num * a340_200_cost

# Airbus A340 Model 300
a340_300_num = 24
a340_300_cost = 88.5
daily_a340_300_flights = 2
a340_300_flights = a340_300_num * daily_a340_300_flights * operating_days
a340_300_value = a340_300_num * a340_300_cost

# Calculate totals
total_aircraft = boe_num + a340_200_num + a340_300_num
total_flights = boe_flights + a340_200_flights + a340_300_flights
total_value = boe_value + a340_200_value + a340_300_value
# Results
print("Total aircraft:", total_aircraft)
print("Annual flight count:", total_flights)
print("Total replacement value: $", round(total_value, 2), "million")


Total aircraft: 86
Annual flight count: 124402.5
Total replacement value: $ 5958.3 million


The results show that Ontario Gateway has 86 aircraft. The fleet includes three aircraft models: Boeing 757 Model 200, Airbus A340 Model 200, and Airbus A340 Model 300. Given that the average operating days per year for each aircraft is 342 days, the calculated annual flight count is about 124,402.5, and the total replacement value of the fleet is about $5,958.3 million.


## Aircraft Accident Simulation Model

### Simulate aircraft accidents


In [18]:
# Author: Zixiao Ma
acc_rate = 1 / 5000000
expected_acc = acc_rate * total_flights
N = 10000
annual_acc_counts = []
annual_crash_losses = []
five_year_losses = []
for sim in range(N):
    losses_for_five_years = []
    for year in range(5):
        accidents = np.random.poisson(expected_acc)
        annual_acc_counts.append(accidents)

        total_loss = 0
        for acc in range(accidents):
            # Randonly select aircrafts involved
            aircraft = np.random.random()
            if aircraft < boe_flights / total_flights:
                total_loss = total_loss + boe_cost

            elif aircraft < (boe_flights + a340_200_flights) / total_flights:
                total_loss = total_loss + a340_200_cost

            else:
                total_loss = total_loss + a340_300_cost
        annual_crash_losses.append(total_loss)
        losses_for_five_years.append(total_loss)
    five_year_losses.append(losses_for_five_years)
# Display simulation
print("Expected accidents per year:", expected_acc)
print("Annual accident observations:", len(annual_acc_counts))
print("Annual crash loss observations:", len(annual_crash_losses))


Expected accidents per year: 0.0248805
Annual accident observations: 50000
Annual crash loss observations: 50000


The Poisson distribution can be used to model the number of rare events within a fixed period. Therefore, we can use it to simulate the number of accidents in one year because aircraft accidents are rare events. The expected annual number of accidents is calculated by multiplying the given accident rate (1/5000000) by the annual flight count, which gives lambda (0.0249). This model randomly generates the number of accidents for each simulated year. Because these three aircraft models have different replacement costs, we then use a random number between 0 and 1 to select the aircraft model involved in each accident. The selection probabilities are based on each aircraft model's annual flight count. Assuming each aircraft has the same accident chance, aircraft models with more flights are more likely to be involved in an accident. For example, Boeing 757 aircraft account for about 77.53% of all annual flights. We will assign random numbers less than 0.7753 to Boeing 757 aircraft. After selecting the aircraft model, the corresponding replacement cost will be added to the total annual crash costs for that model. This method allows us to estimate financial losses from aircraft accidents involving different aircraft models. We simulate 10,000 five-year periods, which gives 50,000 simulated years. The model stores the annual losses and groups every five years together.


### Summarize the simulation results


In [19]:
# Author: Zixiao Ma
# average annual accidents
total_accidents = sum(annual_acc_counts)
average_accidents = total_accidents / len(annual_acc_counts)
# Average annual crash loss
total_simulated_loss = sum(annual_crash_losses)
average_annual_loss = total_simulated_loss / len(annual_crash_losses)
# Count years with at least one accident
years_with_accident = 0
for accident in annual_acc_counts:
    if accident > 0:
        years_with_accident = years_with_accident + 1
# The probability of years with an accident
prob_of_acc = years_with_accident / len(annual_acc_counts) * 100
# Display results
print("Average accidents per year:", round(average_accidents, 4))
print("Average annual crash loss: $", round(average_annual_loss, 2), "million")
print("Probability of years with an accident:", round(prob_of_acc, 2), "%")

Average accidents per year: 0.0244
Average annual crash loss: $ 1.52 million
Probability of years with an accident: 2.41 %


The result of each run may be different because the simulation uses random numbers. Therefore, a fixed random seed is used to ensure the result is reproducible. For this run, the average annual accident count is 0.0249, and the estimated probability of at least one accident in a year is about 2.41%. Based on these conditions, the average annual crash loss is about $1.52 million.


## Four Insurance Plans

### Annual Risk Analysis

### RCNC1


In [20]:
# Author: Zixiao Ma
rcnc1_fixed_premium = total_value * 0.0045
rcnc1_annual_costs = []

for loss in annual_crash_losses:
    retained_loss = loss * 0.1
    annual_cost = rcnc1_fixed_premium + retained_loss
    rcnc1_annual_costs.append(annual_cost)
average_rcnc1_cost = sum(rcnc1_annual_costs) / len(rcnc1_annual_costs)

# Count years exceeding the risk limits
over_37 = 0
over_41 = 0
for cost in rcnc1_annual_costs:
    if cost > 37:
        over_37 += 1
    if cost > 41:
        over_41 += 1
# Calculate the probability that years exceed the risk limits
rcnc1_over_37_prob = over_37 / len(rcnc1_annual_costs) * 100
rcnc1_over_41_prob = over_41 / len(rcnc1_annual_costs) * 100
# Results
print("RCNC1: ")
print("Average annual cost: $", round(average_rcnc1_cost, 2), "million")
print("Probability above $37 million:", round(rcnc1_over_37_prob, 2), "%")
print("Probability above $41 million:", round(rcnc1_over_41_prob, 2), "%")

RCNC1: 
Average annual cost: $ 26.96 million
Probability above $37 million: 0.03 %
Probability above $41 million: 0.01 %


### RCNC2


In [21]:
# Author: Zixiao Ma
rcnc2_fixed_premium = total_value * 0.001
rcnc2_variable_cap = total_value * 0.01
rcnc2_annual_costs = []

for loss in annual_crash_losses:
    variable_premium = loss * 0.90
    if variable_premium > rcnc2_variable_cap:
        variable_premium = rcnc2_variable_cap

    annual_cost = rcnc2_fixed_premium + variable_premium
    rcnc2_annual_costs.append(annual_cost)
average_rcnc2_cost = sum(rcnc2_annual_costs) / len(rcnc2_annual_costs)

# Count years exceeding the risk limits
over_37 = 0
over_41 = 0
for cost in rcnc2_annual_costs:
    if cost > 37:
        over_37 += 1
    if cost > 41:
        over_41 += 1
# Calculate the probability that years exceed the risk limits
rcnc2_over_37_prob = over_37 / len(rcnc2_annual_costs) * 100
rcnc2_over_41_prob = over_41 / len(rcnc2_annual_costs) * 100
# Results
print("RCNC2: ")
print("Average annual cost: $", round(average_rcnc2_cost, 2), "million")
print("Probability above $37 million:", round(rcnc2_over_37_prob, 2), "%")
print("Probability above $41 million:", round(rcnc2_over_41_prob, 2), "%")

RCNC2: 
Average annual cost: $ 7.23 million
Probability above $37 million: 2.41 %
Probability above $41 million: 2.41 %


### CTC


In [22]:
# Author: Zixiao Ma
ctc_fixed_premium = 13
ctc_annual_costs = []

for loss in annual_crash_losses:
    if loss <= 80:
        retained_loss = loss * 0.10
    else:
        retained_loss = 80 * 0.10 + (loss - 80)
    annual_cost = ctc_fixed_premium + retained_loss
    ctc_annual_costs.append(annual_cost)

average_ctc_cost = sum(ctc_annual_costs) / len(ctc_annual_costs)

# Count years exceeding the risk limits
over_37 = 0
over_41 = 0
for cost in ctc_annual_costs:
    if cost > 37:
        over_37 += 1
    if cost > 41:
        over_41 += 1
# Calculate the probability that years exceed the risk limits
ctc_over_37_prob = over_37 / len(ctc_annual_costs) * 100
ctc_over_41_prob = over_41 / len(ctc_annual_costs) * 100
# Results
print("CTC: ")
print("Average annual cost: $", round(average_ctc_cost, 2), "million")
print("Probability above $37 million:", round(ctc_over_37_prob, 2), "%")
print("Probability above $41 million:", round(ctc_over_41_prob, 2), "%")

CTC: 
Average annual cost: $ 13.19 million
Probability above $37 million: 0.03 %
Probability above $41 million: 0.03 %


### HIC


In [23]:
# Author: Zixiao Ma
hic_fixed_premium = total_value * 0.00165
hic_annual_costs = []

for loss in annual_crash_losses:
    if loss <= 24:
        retained_loss = loss
    else:
        retained_loss = 24
    annual_cost = hic_fixed_premium + retained_loss
    hic_annual_costs.append(annual_cost)
average_hic_cost = sum(hic_annual_costs) / len(hic_annual_costs)

# Count years exceeding the risk limits
over_37 = 0
over_41 = 0
for cost in hic_annual_costs:
    if cost > 37:
        over_37 += 1
    if cost > 41:
        over_41 += 1
# Calculate the probability that years exceed the risk limits
hic_over_37_prob = over_37 / len(hic_annual_costs) * 100
hic_over_41_prob = over_41 / len(hic_annual_costs) * 100
# Results
print("HIC: ")
print("Average annual cost: $", round(average_hic_cost, 2), "million")
print("Probability above $37 million:", round(hic_over_37_prob, 4), "%")
print("Probability above $41 million:", round(hic_over_41_prob, 4), "%")

HIC: 
Average annual cost: $ 10.41 million
Probability above $37 million: 0.0 %
Probability above $41 million: 0.0 %


This part compares the annual costs of the four insurance plans. The costs include insurance premiums and the losses or variable premiums paid by Ontario Gateway under each plan. It also calculates the probability that the annual cost exceeds ＄37 million and ＄41 million. ＄41 million is the absolute maximum threshold, and ＄37 million is the target risk threshold.

RCNC1 has the highest average annual cost at $26.96 million. Its probability of exceeding ＄37 million and ＄41 million is 0.03% and 0.01%, respectively. This shows that RCNC1 has a high annual cost, but these risk events are very rare.

RCNC2 has the lowest average cost at $7.23 million. However, its probability of exceeding both ＄37 million and ＄41 million is 2.41%. Even though the annual cost of RCNC2 is relatively low, the probability of these risk events is relatively high.

CTC has an average annual cost of ＄13.19 million, and its probability of exceeding either risk threshold is 0.03%. Its average annual cost is lower than that of RCNC1. However, its probability of exceeding ＄37 million is the same as RCNC1, and its probability of exceeding $41 million is higher.

HIC has an average annual cost of $10.41 million. Since HIC states that the company pays a maximum of ＄24 million in annual retained aircraft losses, the maximum annual cost is approximately ＄33.83 million (5958.3\*0.00165+24). Therefore, its annual cost is impossible to exceed＄37 million.


### Five-Year Total Cost Analysis


In [24]:
# Author: Zixiao Ma
# Create lists to store the five-year costs
rcnc1_five_year_costs = []
rcnc2_five_year_costs = []
ctc_five_year_costs = []
hic_five_year_costs = []

# Create lists to store rebates
rcnc1_rebates = []
hic_rebates = []

# Calculate the five-year costs
for sim, five_year in enumerate(five_year_losses):
    start = sim * 5
    end = start + 5

    total_loss = sum(five_year)
    # RCNC1
    cost_before_rebate = sum(rcnc1_annual_costs[start:end])
    insurer_profit = 5 * rcnc1_fixed_premium - total_loss * 0.90
    rebate = max(0, insurer_profit) * 0.20
    rcnc1_five_year_costs.append(cost_before_rebate - rebate)
    rcnc1_rebates.append(rebate)

    # RCNC2
    total_cost = sum(rcnc2_annual_costs[start:end])
    rcnc2_five_year_costs.append(total_cost)

    # CTC
    total_cost = sum(ctc_annual_costs[start:end])
    ctc_five_year_costs.append(total_cost)

    # HIC
    cost_before_rebate = sum(hic_annual_costs[start:end])
    insurer_claims = 0

    for loss in five_year:
        insurer_claims += max(loss - 24, 0)
    insurer_profit = 5 * hic_fixed_premium - insurer_claims
    rebate = max(0, insurer_profit) * 0.035
    hic_five_year_costs.append(cost_before_rebate - rebate)
    hic_rebates.append(rebate)

# Calculate average five-year net costs
results = pd.DataFrame(
    {
        "Insurance Plan": ["RCNC1", "RCNC2", "CTC", "HIC"],
        "Average Five-Year Net Cost ($ million)": [
            sum(rcnc1_five_year_costs) / len(rcnc1_five_year_costs),
            sum(rcnc2_five_year_costs) / len(rcnc2_five_year_costs),
            sum(ctc_five_year_costs) / len(ctc_five_year_costs),
            sum(hic_five_year_costs) / len(hic_five_year_costs),
        ],
    }
)

results = results.sort_values("Average Five-Year Net Cost ($ million)")
results.round(2)

,Insurance Plan,Average Five-Year Net Cost ($ million)
1,RCNC2,36.16
3,HIC,50.48
2,CTC,65.93
0,RCNC1,109.38


The simulated results show that RCNC2 has the lowest average five-year net cost at ＄36.16 million, and followed by HIC with a cost of ＄50.48 million. The average five-year net cost of CTC is ＄65.93 million, and RCNC1 has the highest average five-year net cost at ＄109.38 million.

Because RCNC2 and CTC do not include five-year rebates, their average five-year costs are approximately equal to their average annual costs multiplied by 5. For RCNC1 and HIC, their calculation also considers the rebates specified in their insurance contracts.


## Overall Comparison and Recommendation


In [25]:
# Author: Zixiao Ma
# Add annual risk probabilities
results["Annual Probability > $37M (%)"] = results["Insurance Plan"].map(
    {
        "RCNC1": rcnc1_over_37_prob,
        "RCNC2": rcnc2_over_37_prob,
        "CTC": ctc_over_37_prob,
        "HIC": hic_over_37_prob,
    }
)
results["Annual Probability > $41M (%)"] = results["Insurance Plan"].map(
    {
        "RCNC1": rcnc1_over_41_prob,
        "RCNC2": rcnc2_over_41_prob,
        "CTC": ctc_over_41_prob,
        "HIC": hic_over_41_prob,
    }
)
results = results.sort_values("Average Five-Year Net Cost ($ million)")
results.round(2)

,Insurance Plan,Average Five-Year Net Cost ($ million),Annual Probability > $37M (%),Annual Probability > $41M (%)
1,RCNC2,36.16,2.41,2.41
3,HIC,50.48,0.00,0.00
2,CTC,65.93,0.03,0.03
0,RCNC1,109.38,0.03,0.01


Based on the simulated results, HIC is the best chocie when both cost and risk are considered. Although it is not the cheapest option, its costs is impossible to exceed the $37 million target threshold. Considering that Ontario Gateway has high debt and needs to maintain stable cash flow, reducing financial risk is very important. HIC can provide better protection against large annual costs. Therefore, we recommend HIC.


## Complete the Baseline Simulation

**Units:** all monetary amounts are in millions of US dollars.

**Assumptions:**

- Each year has 342 operating days. Fleet size and replacement costs remain constant over five years; lost aircraft are replaced.
- Crashes are rare, so annual crash counts for each aircraft model follow a Poisson distribution. Its mean equals that model's annual flights multiplied by the per-flight crash probability.
- Annual incidental damage follows a uniform distribution between \$1M and \$5M. This is an explicit modeling assumption because the case supplies a range, but no distribution.
- Total annual aircraft loss includes crash losses and incidental damage. We assume both are covered under the stated plan formulas. This coverage interpretation should be checked if more detailed policy wording becomes available.
- Years and aircraft-model crash counts are independent. Incidental damage is independent of crashes. No discounting, inflation, or fleet growth is assumed.
- All four plans use the same loss scenarios, making the cost comparison fair.

**Calculation:** annual company cost = premium + loss not paid by the insurer. RCNC2 fully covers losses, with an additional premium capped at 1% of fleet value. Five-year rebates are calculated from positive cumulative insurer profit and are paid at the end of year five; they do not reduce first-year costs.


In [31]:
# Author: Haolin Zhang
planes = np.array([47, 15, 24])
replacement_cost = np.array([56.42, 78.94, 88.55])
flights_per_day = np.array([6, 2.25, 2])
annual_flights = planes * flights_per_day * 342
fleet_value = sum(planes * replacement_cost)
number_of_trials = 100000
industry_crash_probability = 1 / 5000000

print("Fleet value ($M):", round(fleet_value, 2))
print("Annual flights:", sum(annual_flights))
print("Five-year trials:", number_of_trials)

Fleet value ($M): 5961.04
Annual flights: 124402.5
Five-year trials: 100000


### One reusable simulation function

The function below avoids repeating the same calculations in Part C. Each row represents one five-year scenario, and each column represents a year. NumPy performs the same simple arithmetic across all scenarios at once.


In [32]:
# Author: Haolin Zhang
def simulate_insurance(crash_probability, seed=123):
    rng = np.random.default_rng(seed)
    crash_losses = np.zeros((number_of_trials, 5))

    for model in range(3):
        expected_crashes = annual_flights[model] * crash_probability
        crashes = rng.poisson(expected_crashes, size=(number_of_trials, 5))
        crash_losses += crashes * replacement_cost[model]

    incidental_damage = rng.uniform(1, 5, size=(number_of_trials, 5))
    losses = crash_losses + incidental_damage

    annual_cost = {}
    annual_cost["RCNC1"] = fleet_value * 0.0045 + losses * 0.10
    annual_cost["RCNC2"] = fleet_value * 0.001 + np.minimum(
        losses * 0.90, fleet_value * 0.01
    )
    annual_cost["CTC"] = 13 + losses - 0.90 * np.minimum(losses, 80)
    annual_cost["HIC"] = fleet_value * 0.00165 + np.minimum(losses, 24)

    rcnc1_profit = 5 * fleet_value * 0.0045 - (0.90 * losses).sum(axis=1)
    hic_profit = 5 * fleet_value * 0.00165 - np.maximum(losses - 24, 0).sum(axis=1)
    rcnc1_rebate = 0.20 * np.maximum(rcnc1_profit, 0)
    hic_rebate = 0.035 * np.maximum(hic_profit, 0)

    five_year_cost = {}
    rows = []
    for plan in annual_cost:
        five_year_cost[plan] = annual_cost[plan].sum(axis=1)
        if plan == "RCNC1":
            five_year_cost[plan] -= rcnc1_rebate
        if plan == "HIC":
            five_year_cost[plan] -= hic_rebate

        first_year = annual_cost[plan][:, 0]
        rows.append(
            {
                "Plan": plan,
                "Mean first-year cost ($M)": first_year.mean(),
                "First-year P(>$37M) (%)": 100 * (first_year > 37).mean(),
                "First-year P(>$41M) (%)": 100 * (first_year > 41).mean(),
                "Mean five-year net cost ($M)": five_year_cost[plan].mean(),
                "Five-year cost SD ($M)": five_year_cost[plan].std(ddof=1),
            }
        )

    summary = pd.DataFrame(rows).set_index("Plan")
    return summary, five_year_cost, annual_cost


baseline_summary, baseline_costs, baseline_annual = simulate_insurance(
    industry_crash_probability
)
print(baseline_summary.round(4).to_string())

       Mean first-year cost ($M)  First-year P(>$37M) (%)  First-year P(>$41M) (%)  Mean five-year net cost ($M)  Five-year cost SD ($M)
Plan                                                                                                                                    
RCNC1                    27.2837                    0.038                    0.007                      113.6778                  6.3060
RCNC2                     9.9622                    2.486                    2.486                       49.7171                 18.2668
CTC                      13.5144                    0.038                    0.038                       67.5362                  3.8867
HIC                      13.3591                    0.000                    0.000                       65.1993                  8.1255


### Check financial risk and simulation precision

A simulated zero exceedance rate alone cannot guarantee safety. Here HIC has a contractual annual cost ceiling: its premium plus \$24M retained loss. The other plans can exceed \$41M in sufficiently severe scenarios, even if those events are rare. The case does not specify an acceptable positive probability of exceeding the absolute limit, so we distinguish strict feasibility from plans that management might accept under a less strict interpretation.


In [33]:
# Author: Haolin Zhang
hic_maximum_cost = fleet_value * 0.00165 + 24
print("HIC contractual maximum annual cost ($M):", round(hic_maximum_cost, 4))
print("HIC margin below the $37M target ($M):", round(37 - hic_maximum_cost, 4))

risk_rows = []
for plan in baseline_annual:
    over_limit = baseline_annual[plan][:, 0] > 41
    probability = over_limit.mean()
    standard_error = np.sqrt(probability * (1 - probability) / number_of_trials)
    risk_rows.append(
        {
            "Plan": plan,
            "First-year events >$41M": int(over_limit.sum()),
            "Probability (%)": 100 * probability,
            "Monte Carlo SE (percentage points)": 100 * standard_error,
        }
    )
print(pd.DataFrame(risk_rows).set_index("Plan").round(5).to_string())

assert hic_maximum_cost < 37
assert all(len(values) == number_of_trials for values in baseline_costs.values())
assert all(np.isfinite(values).all() for values in baseline_costs.values())

HIC contractual maximum annual cost ($M): 33.8357
HIC margin below the $37M target ($M): 3.1643
       First-year events >$41M  Probability (%)  Monte Carlo SE (percentage points)
Plan                                                                               
RCNC1                        7            0.007                             0.00265
RCNC2                     2486            2.486                             0.04924
CTC                         38            0.038                             0.00616
HIC                          0            0.000                             0.00000


(b) To add clarity to the analysis, modify the model so that you can analyze the cost
savings between the plans that are viable.


Under a literal interpretation of the $41M absolute maximum, **only HIC is strictly viable**, so there is no pair of strictly viable plans to compare. We therefore also show HIC versus RCNC1 and CTC as conditional comparisons if management accepts their small tail risks. RCNC2 is shown for context, not classified as viable.

For each five-year scenario, define **savings from HIC = other plan's net cost − HIC's net cost**. A positive number means HIC is cheaper. Taking differences within the same loss scenario separates plan differences from differences in simulated accidents.


In [34]:
# Author: Haolin Zhang
def compare_with_hic(five_year_cost):
    rows = []
    for other_plan in ["RCNC1", "CTC", "RCNC2"]:
        savings = five_year_cost[other_plan] - five_year_cost["HIC"]
        rows.append(
            {
                "Comparison": other_plan + " minus HIC",
                "Mean savings from HIC ($M)": savings.mean(),
                "Savings SD ($M)": savings.std(ddof=1),
                "5th percentile ($M)": np.percentile(savings, 5),
                "95th percentile ($M)": np.percentile(savings, 95),
                "P(HIC is cheaper) (%)": 100 * (savings > 0).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("Comparison")


baseline_savings = compare_with_hic(baseline_costs)
print(baseline_savings.round(4).to_string())

                 Mean savings from HIC ($M)  Savings SD ($M)  5th percentile ($M)  95th percentile ($M)  P(HIC is cheaper) (%)
Comparison                                                                                                                    
RCNC1 minus HIC                     48.4785           2.7431              42.8190               52.1239                100.000
CTC minus HIC                        2.3369           5.8919             -12.3973                7.7964                 85.700
RCNC2 minus HIC                    -15.4822          10.4266             -19.5662               10.3847                 11.659


The mean estimates expected five-year savings. The 5th and 95th percentiles describe the middle 90% of simulated savings outcomes; they are not a confidence interval for the mean. The probability column shows how often HIC costs less in matched scenarios. Savings must be considered together with Part A's cash-flow limits: a cheaper plan may still expose the airline to unacceptable annual costs.


(c) Check the sensitivity of your analysis to the presumed probability of a crash. Be-
cause Ontario Gateway operates a newer fleet of aircraft with only two different
types of aircraft, it may be reasonable to suppose that they are slightly safer than
the industry as whole. Change the probability of a crash to make Ontario Gate-
way 25% safer than the industry average, and see what effect this has on the out-
put and the consequent insurance decision.


“25% safer” is interpreted as a **25% reduction in per-flight crash probability**, so the revised probability is `0.75 / 5,000,000`. All other assumptions stay the same. We reuse the function and the seed to keep the procedure reproducible. Reusing a seed does not guarantee perfectly matched scenarios between the two crash-rate settings because Poisson sampling can consume random numbers differently.


In [35]:
# Author: Haolin Zhang
safer_crash_probability = industry_crash_probability * 0.75
safer_summary, safer_costs, safer_annual = simulate_insurance(safer_crash_probability)
print("Original per-flight crash probability:", industry_crash_probability)
print("Safer per-flight crash probability:", safer_crash_probability)
print(
    "Expected crashes per year, original:",
    sum(annual_flights) * industry_crash_probability,
)
print(
    "Expected crashes per year, safer:", sum(annual_flights) * safer_crash_probability
)
print("\nSafer scenario:")
print(safer_summary.round(4).to_string())

comparison = pd.DataFrame(index=baseline_summary.index)
comparison["Baseline five-year mean ($M)"] = baseline_summary[
    "Mean five-year net cost ($M)"
]
comparison["Safer five-year mean ($M)"] = safer_summary["Mean five-year net cost ($M)"]
comparison["Change in five-year mean ($M)"] = (
    comparison["Safer five-year mean ($M)"] - comparison["Baseline five-year mean ($M)"]
)
comparison["Baseline P(>$37M) (%)"] = baseline_summary["First-year P(>$37M) (%)"]
comparison["Safer P(>$37M) (%)"] = safer_summary["First-year P(>$37M) (%)"]
comparison["Baseline P(>$41M) (%)"] = baseline_summary["First-year P(>$41M) (%)"]
comparison["Safer P(>$41M) (%)"] = safer_summary["First-year P(>$41M) (%)"]
print("\nBaseline versus safer scenario:")
print(comparison.round(4).to_string())
print("\nCost savings under the safer scenario:")
print(compare_with_hic(safer_costs).round(4).to_string())

Original per-flight crash probability: 2e-07
Safer per-flight crash probability: 1.5e-07
Expected crashes per year, original: 0.0248805
Expected crashes per year, safer: 0.018660375

Safer scenario:
       Mean first-year cost ($M)  First-year P(>$37M) (%)  First-year P(>$41M) (%)  Mean five-year net cost ($M)  Five-year cost SD ($M)
Plan                                                                                                                                    
RCNC1                    27.2387                    0.023                    0.005                      113.1287                  5.4762
RCNC2                     9.5956                    1.789                    1.789                       48.1009                 15.8521
CTC                      13.4523                    0.023                    0.023                       67.2686                  3.2645
HIC                      13.2108                    0.000                    0.000                       64.5109    

Lower crash probability reduces expected crash losses, but premiums, deductibles, caps, and rebates determine how strongly each plan's cost changes. Compare the tables rather than assuming total cost falls by 25%; fixed premiums and incidental damage do not fall. HIC's contractual ceiling is unchanged, so it remains the only strictly viable plan under the absolute-limit interpretation. Lower crash probability does not remove the other plans' possible extreme losses. Small changes in rare exceedance rates may reflect Monte Carlo noise. This addresses the sensitivity of the insurance decision; the recommendation is given in Part D below.


(d) Based on your simulation results and analysis/judgment, prepare a concise but
detailed decision recommendation with supporting analysis and justification.
Your recommendation should account for the nature of the uncertainty and risk
involved in the decisions under consideration.
